# E3a-c: where the preamble acts (HF Jobs L4, bf16, eager attention, fit split only)

In [ ]:
!git clone -q -b feat/hsafe https://github.com/dhairya-pandya/Lexsi-AI-Alignment repo
%cd repo
!pip install -q -e .

In [ ]:
import json, yaml, numpy as np, pandas as pd, torch
from hsafe.interp.hooks import load_model, full_attention_layers
from hsafe.interp.attention import attention_mass
from hsafe.interp.patching import patch_layers
from hsafe.interp.directions import refusal_direction, projections
from hsafe.records import read_jsonl, PromptRecord
from hsafe.render import render
m = yaml.safe_load(open('configs/models.yaml'))['tiny-aya-global']
model, tok = load_model(m['id'], m['revision'], torch.bfloat16, eager=True)
openers = json.load(open('results/gate_g1/openers.json'))
harm = [r for r in read_jsonl('data/prompts/harmful_en.jsonl', PromptRecord) if r.split == 'fit' and r.bucket != 'ACCEPT'][:40]
ben = [r for r in read_jsonl('data/prompts/benign.jsonl', PromptRecord) if r.split == 'fit'][:40]
FULL = full_attention_layers(model.config); SLIDING = [i for i in range(model.config.num_hidden_layers) if i not in FULL]

In [ ]:
# (a) attention mass request -> preamble, by distance
import os; os.makedirs('results/e3', exist_ok=True)
mass = {}
for d in [1000, 3900, 4100, 6000]:
    rs = [render(r, 'P-default', d, 'F2', tok, 'tiny_aya') for r in harm[:16]]
    mass[d] = attention_mass(model, rs)
np.savez('results/e3/attention.npz', **{str(k): v for k, v in mass.items()})

In [ ]:
# (b) patching P-default -> P-none at the request positions
rows = []
for r in harm:
    src = render(r, 'P-default', None, None, tok, 'tiny_aya'); tgt = render(r, 'P-none', None, None, tok, 'tiny_aya')
    for comp in ['attn', 'resid']:
        for name, layers in [('none', []), ('full', FULL), ('sliding', SLIDING), ('all', FULL + SLIDING)]:
            rows.append(dict(record_id=r.id, component=comp, layers=name, p_refusal=patch_layers(model, src, tgt, layers, comp, openers)))
pd.DataFrame(rows).to_csv('results/e3/patching.csv', index=False)
pd.DataFrame(rows).groupby(['component','layers']).p_refusal.mean()

In [ ]:
# (c) per-layer projection onto the refusal direction under each condition
L = 24  # cross-check with SafeTune extract_refusal_direction's sweep
fit_h = [render(r, 'P-default', None, None, tok, 'tiny_aya') for r in harm]
fit_b = [render(r, 'P-default', None, None, tok, 'tiny_aya') for r in ben]
direction = refusal_direction(model, fit_h, fit_b, L)
torch.save(direction, 'results/e3/direction_pdefault_L24.pt')
proj = {c: projections(model, [render(r, c, None, None, tok, 'tiny_aya') for r in harm], direction) for c in ['P-default', 'P-none']}
proj['P-default@6000'] = projections(model, [render(r, 'P-default', 6000, 'F2', tok, 'tiny_aya') for r in harm], direction)
np.savez('results/e3/projections.npz', **proj)